In [4]:
import kfp
from kfp import dsl
from kfp import kubernetes
from kfp import local
from kfp.dsl import Input, Output, Dataset, Model, Artifact

# TIP: you may need to authenticate with the KFP instance
# local.init(runner=local.SubprocessRunner())
kfp_client = kfp.Client()

In [2]:
current_sc = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.spec.storageClassName}'").read()
namespace_cur = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.metadata.namespace}'").read()
print(namespace_cur)
print(current_sc)

geun-tak-roh-2e590eb8
gl4f-filesystem


In [22]:
import os
import base64
import kserve
from kubernetes import client,config,utils
from kubernetes.client.rest import ApiException

config.load_incluster_config()
v1 = client.CoreV1Api()
custom_api = client.CustomObjectsApi()

secret_name = 'my-secret'
namespace = namespace_cur

with open('/etc/secrets/ezua/.auth_token','r') as file:
    AUTH_TOKEN = file.read().strip()

try:
    ## Check pre-exists secret
    preexists_secret = v1.read_namespaced_secret(namespace=namespace,name=secret_name)
    print(f"secret name : {preexists_secret.metadata.name} exists")
    v1.delete_namespaced_secret(namespace=namespace,name=secret_name)

except ApiException as e:
    print("Exception when calling CoreV1Api : %s\n" % e)

finally:
    ## Create secret snippet start
    secret_data_encoded = {
        "AWS_ACCESS_KEY_ID": base64.b64encode(AUTH_TOKEN.encode()).decode(),
        "AWS_SECRET_ACCESS_KEY": base64.b64encode("s3".encode()).decode(),
        "AWS_REGION": base64.b64encode("local".encode()).decode(),
    }
    
    secret_body = client.V1Secret(
        api_version="v1",
        kind="Secret",
        metadata=client.V1ObjectMeta(
            name=secret_name,
            annotations={
                "serving.kserve.io/s3-cabundle":"",
                "serving.kserve.io/s3-endpoint":"local-s3-service.ezdata-system.svc.cluster.local:30000/",
                "serving.kserve.io/s3-useanoncredential":"false",
                "serving.kserve.io/s3-usehttps":"0",
                "serving.kserve.io/s3-verifyssl":"0",
            }
        ),
        type="Opaque",
        data=secret_data_encoded,  # Use data or string_data
    )
    
    api_response = v1.create_namespaced_secret(namespace=namespace, body=secret_body)
    print(f"Secret '{api_response.metadata.name}' created successfully in namespace '{namespace}'.\n api_response:{api_response}")
    ## Create secret snippet end

secret name : my-secret exists
Secret 'my-secret' created successfully in namespace 'geun-tak-roh-2e590eb8'.
 api_response:{'api_version': 'v1',
 'data': {'AWS_ACCESS_KEY_ID': 'ZXlKaGJHY2lPaUpTVXpJMU5pSXNJblI1Y0NJZ09pQWlTbGRVSWl3aWEybGtJaUE2SUNJM1pUVlZRMGwxVFRWdmQzazNjSEJCYmpsT1pXdEtkbEkxZEVWa01YUjVTakp3WDFFMWFWSnVhRzFCSW4wLmV5SmxlSEFpT2pFM05UUXlPRE0zTXpJc0ltbGhkQ0k2TVRjMU5ESTRNVGt6TWl3aVlYVjBhRjkwYVcxbElqb3hOelUwTWpBek16ZzVMQ0pxZEdraU9pSmlZbUk1WVRkaU9TMW1aR1kxTFRRMk1Ea3RPVGcyWWkwd1pqVTRNemt4WW1Wa1lXSWlMQ0pwYzNNaU9pSm9kSFJ3Y3pvdkwydGxlV05zYjJGckxtbHVaM0psYzNNdWNHTmhhVEF6TURndWMyY3lMbWh3WldOdmJHOHVibVYwTDNKbFlXeHRjeTlWUVNJc0luTjFZaUk2SW1KaU16bGtPV0k0TFdKbE9HWXRORFF5TUMxaVpqUmhMV0k0WTJVMk9EZ3hZMkkwWVNJc0luUjVjQ0k2SWtKbFlYSmxjaUlzSW1GNmNDSTZJblZoSWl3aWJtOXVZMlVpT2lKb1FVbzFRVWxvYTJOdlpGQjRVM0pNUW1GaVkxTlpXRmxIV0ZwRVF6ZFRkVXRTYURWUUxYWjZRakl3SWl3aWMyVnpjMmx2Ymw5emRHRjBaU0k2SWpjNU1EQTVaVGRqTFRRNE1Ua3ROR1psTUMwNFlUY3lMVEF6TWpoaFlqWTJZVEUyTWlJc0ltRmpjaUk2SWpFaUxDSnpZMjl3WlNJNkltOXdaVzVwWkNCbGJ

In [23]:
@dsl.component(
    base_image='geuntakroh/kfp-test:v0.9',
)
def write_artifacts(output_artifacts: Output[Artifact]):
    import subprocess
    subprocess.run(['env'])
    print(output_artifacts.path)
    with open(output_artifacts.path,'w') as output_file:
        output_file.write("test artifacts")

In [34]:
@dsl.pipeline(
    name="prepare_model_pipe"
)
def prepare_model_pipe():
    task1 = write_artifacts()
    kfp.kubernetes.use_secret_as_env(
        task=task1,
        secret_name='my-secret',
        secret_key_to_env={'AWS_SECRET_ACCESS_KEY': 'AWS_SECRET_ACCESS_KEY'}
    )
    kubernetes.use_secret_as_env(
        task=task1,
        secret_name='my-secret',
        secret_key_to_env={'AWS_ACCESS_KEY_ID': 'AWS_ACCESS_KEY_ID'}
    )
    kubernetes.use_secret_as_env(
        task=task1,
        secret_name='my-secret',
        secret_key_to_env={'AWS_REGION': 'AWS_REGION'}
    )
    task1.set_env_variable(
        name='AWS_ENDPOINT_URL_S3',
        value='http://local-s3-service.ezdata-system.svc.cluster.local:30000'
    )

In [35]:
kfp_client.create_run_from_pipeline_func(
    prepare_model_pipe,
    experiment_name="test-rhgt-exp",
    enable_caching=False,
    pipeline_root=''
)

RunPipelineResult(run_id=56136fdd-2fa4-4e4b-b495-04174d28df7c)